# Práctica 4: Embeddings distribucionales y contextuals

## Introducción Teórica y Objetivos
El objetivo de la práctica es entrenar y evaluar modelos de **embeddings distribucionales (estáticos)** y compararlos con **embeddings contextuales** para tareas de similitud semántica.

Los **Word Embeddings** permiten representar las palabras como vectores numéricos basándose en su contexto.
1. **Estáticos (Word2Vec, fastText)**: Cada palabra tiene siempre el mismo vector. Compararemos el comportamiento frente a palabras fuera del vocabulario (OOV) y el efecto del tamaño del vector y del corpus de entrenamiento.
2. **Contextuales (BERT, RoBERTa)**: Generan representaciones dinámicas basadas en la oración entera, lo que resuelve problemas como la polisemia.



## Evaluación
* **Intrínseca (Multi-SimLex)**: Evaluaremos la distancia coseno entre vectores asociados para obtener la similitud y calcularemos la **correlación de Spearman** comparada con evaluadores humanos.
* **Extrínseca (Spanish STS)**: Calcularemos la similitud semántica de documentos enteros (frases) y utilizaremos la **correlación de Pearson**.



## Datasets
- **Multi-SimLex (Spanish):** conjunto de pares de palabras con puntuaciones de similitud semántica anotadas por humanos. Se utiliza para la **evaluación intrínseca**, midiendo directamente la calidad de los embeddings mediante la **correlación de Spearman** entre las similitudes predichas y las humanas.

- **Spanish STS:** conjunto de pares de frases con puntuaciones de similitud semántica. Se utiliza para la **evaluación extrínseca**, midiendo la **correlación de Pearson** entre las similitudes predichas y las de referencia.

- **Corpus Wikipedia en español (`raw.es.tgz`):** corpus de texto en crudo utilizado para entrenar los embeddings distribucionales.

## 0. Setup

In [23]:
# Instalar dependencias si es necesario
# %pip install pandas scipy datasets tqdm gensim torch sentence-transformers scikit-learn transformers
# pip install -U datasets

import pandas as pd
import numpy as np
import os
import re
from scipy.stats import spearmanr, pearsonr
from datasets import load_dataset
from gensim.models import Word2Vec, FastText
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel

# %pip install requests "datasets<3.0"
# from pathlib import Path
# from tqdm.notebook import tqdm
# import nltk
# from itertools import product
# from sklearn.model_selection import KFold

## 1. Datasets

Esta práctica utiliza tres datasets:

| Dataset | Descripción | Uso |
|---|---|---|
| **Multi-SimLex** | Pares de palabras con puntuaciones de similitud semántica | Evaluación intrínseca |
| **Spanish STS** | Pares de frases con puntuaciones de similitud semántica | Evaluación extrínseca |
| **raw_es** | Corpus de Wikipedia en español | Entrenamiento de embeddings |

A partir de `raw_es`, entrenaremos y compararemos modelos de **Word2Vec** y **fastText**.

## Columnas del dataset Multi-SimLex (ES)

| Columna | Descripción |
|---|---|
| **PAIR_ID** | Identificador único del par de palabras en el dataset. |
| **SPA_W1** | Primera palabra del par. |
| **SPA_W2** | Segunda palabra del par. |
| **SPA_AVG** | Puntuación media de similitud semántica asignada por varios anotadores humanos. Cuanto mayor es el valor, más similares se consideran las dos palabras. En este dataset los valores observados van aproximadamente de **0.0** a **5.9**. |

In [24]:
# ============================================================
# DATASET 1 — Multi-SimLex  (reemplaza la celda original)
# ============================================================

os.system("wget -O SPA.csv https://web.archive.org/web/20231020014354/https://multisimlex.com/data/SPA.csv")

spa = pd.read_csv("SPA.csv")

annotator_cols = [c for c in spa.columns if c.startswith('Annotator')]
spa['SPA_AVG'] = spa[annotator_cols].mean(axis=1)

# Guardamos también las columnas de anotadores para el análisis posterior
multi_simlex = spa[['ID', 'Word 1', 'Word 2'] + annotator_cols + ['SPA_AVG']].copy()
multi_simlex.columns = ['PAIR_ID', 'SPA_W1', 'SPA_W2'] + annotator_cols + ['SPA_AVG']

print(f"Multi-SimLex (ES) — {len(multi_simlex)} pares de palabras")
print(f"Columnas de anotadores: {annotator_cols}")
multi_simlex.head()

Multi-SimLex (ES) — 1888 pares de palabras
Columnas de anotadores: ['Annotator 1', 'Annotator 2', 'Annotator 3', 'Annotator 4', 'Annotator 5', 'Annotator 6', 'Annotator 7', 'Annotator 8', 'Annotator 9', 'Annotator 10']


,PAIR_ID,SPA_W1,SPA_W2,Annotator 1,Annotator 2,Annotator 3,Annotator 4,Annotator 5,Annotator 6,Annotator 7,Annotator 8,Annotator 9,Annotator 10,SPA_AVG
0,1,brazo,músculo,1,1,5,0,0,0,2,1,2,2,1.4
1,2,democracia,monarquía,0,0,3,0,0,0,2,1,3,4,1.3
2,3,tejado,techo,5,5,6,4,5,3,4,4,6,6,4.8
3,4,amigo,profesor,0,0,1,0,0,0,2,0,1,0,0.4
4,5,mano,pie,1,0,3,0,0,0,2,1,3,1,1.1


In [25]:
# ver valores mínimo y máximo de las puntuaciones
min_score = multi_simlex['SPA_AVG'].min()
max_score = multi_simlex['SPA_AVG'].max()

print("Valor mínimo:", min_score)
print("Valor máximo:", max_score)

Valor mínimo: 0.0
Valor máximo: 5.9


## Columnas del dataset Spanish STS

| Columna | Descripción |
|---|---|
| **id** | Identificador único del par de frases. |
| **sentence1** | Primera frase del par. |
| **sentence2** | Segunda frase del par. |
| **score** | Puntuación de similitud semántica asignada por anotadores humanos. Cuanto mayor es el valor, más similares son las dos frases. |

In [26]:
# ============================================================
# DATASET 2 — Spanish STS (PlanTL-GOB-ES/sts-es)
# Avaluación extrínseca · Mètrica: correlación de Pearson
# ============================================================
sts = load_dataset("PlanTL-GOB-ES/sts-es")

train_df = sts["train"].to_pandas().rename(columns={"label": "score"})
dev_df   = sts["validation"].to_pandas().rename(columns={"label": "score"})
test_df  = sts["test"].to_pandas().rename(columns={"label": "score"})

print(f"Train: {len(train_df)} | Dev: {len(dev_df)} | Test: {len(test_df)}")
print(train_df.head())

Train: 1320 | Dev: 77 | Test: 155
  id                                          sentence1  \
0  0  Según el sondeo, 87% de los católicos cree que...   
1  1  La psicología explora conceptos como la percep...   
2  2  La tradición comenzó en el siglo IV, pero la m...   
3  3  "Maria Anna Schwegelin" (? - 1781 en la cárcel...   
4  4  Su identidad la había revelado durante el viaj...   

                                           sentence2  score  
0  El 87% de los católicos del mundo aprobaron el...   3.75  
1  La "psicología básica" es la parte de la psico...   2.80  
2  La tradición de erigir piedras con inscripcion...   2.40  
3  Te entregamos, Anna Schwegelin, al verdugo par...   2.20  
4  La información fue suministrada por el pescado...   2.20  


## Corpus de Wikipedia en español (`raw_es`)

Este corpus se utilizará para entrenar los modelos de embeddings (**Word2Vec** y **fastText**).

El corpus está dividido en múltiples archivos de texto dentro de la carpeta `raw_es`.

Cada archivo contiene una parte del texto extraído de Wikipedia en español.

En total se han encontrado **57 archivos** dentro del corpus.

In [27]:
# ============================================================
# DATASET 3 — Corpus Wikipedia en español
# ============================================================
corpus_dir   = "raw_es"
corpus_files = sorted(os.listdir(corpus_dir))

print(f"Total archivos encontrados: {len(corpus_files)}")
print(corpus_files[:5])

Total archivos encontrados: 57
['spanishText_10000_15000', 'spanishText_110000_115000', 'spanishText_120000_125000', 'spanishText_15000_20000', 'spanishText_180000_185000']


## 2. Preprocesamiento de *raw_es*

Es necesario realizar un preprocesamiento sobre el corpus *raw_es* para eliminar elementos 
que no aportan información: etiquetas **`<doc ...>`/`</doc>`**, **`ENDOFARTICLE`** y **URLs**.

También aplicaremos preprocesamiento básico: convertir a minúsculas, eliminar caracteres no 
alfabéticos y tokenizar.

Respecto a las **stopwords**, la literatura no es concluyente:

- La implementación de Gensim ya realiza *subsampling* automático de palabras frecuentes 
  (parámetro `sample=0.001`), por lo que las stopwords ya reciben menos peso durante el 
  entrenamiento sin necesidad de eliminarlas explícitamente.

- Por otro lado, las stopwords pueden ser útiles para asociar palabras relacionadas a través 
  del contexto sintáctico. Por ejemplo, nombres de ciudades pueden aparecer juntos no solo 
  por verbos como *"ir"* o *"viajar"*, sino también por preposiciones como *"a"*, *"desde"* 
  o *"en"* — relaciones que se perderían al eliminarlas.

- Además, el español tiene una morfología más rica que el inglés (artículos, preposiciones 
  contractas, pronombres clíticos...), por lo que eliminar stopwords puede suponer una 
  pérdida de contexto mayor que en otras lenguas.

Por este motivo, entrenaremos **dos modelos** para cada configuración (con y sin stopwords) 
y evaluaremos empíricamente cuál obtiene mejores resultados en Multi-SimLex y Spanish STS.

**Referencias**:
- Trideep Rath (2016). [*Stopword removing when using the word2vec*](https://stackoverflow.com/questions/34721984/stopword-removing-when-using-the-word2vec). Stack Overflow.
- Jindřich (2020). [*Text preprocessing for text classification using fastText*](https://stackoverflow.com/questions/62244474/text-preprocessing-for-text-classification-using-fasttext). Stack Overflow.

In [28]:
import unicodedata
import re
import os
from nltk.corpus import stopwords

STOPWORDS = set(stopwords.words('spanish'))

def limpiar_texto(texto):
    texto = unicodedata.normalize("NFKC", texto)
    texto = texto.lower()
    texto = re.sub(r'https?://\S+|www\.\S+', '', texto)
    texto = re.sub(r'<[^>]+>', ' ', texto)
    texto = re.sub(r'endofarticle\.?', ' ', texto, flags=re.IGNORECASE)
    texto = re.sub(r'[^a-záéíóúüñ\s]', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

def preprocesar_linea(linea, eliminar_stopwords=True):
    linea = linea.strip()
    if not linea:
        return []
    texto = limpiar_texto(linea)
    palabras = texto.split()
    if eliminar_stopwords:
        palabras = [p for p in palabras if p and p not in STOPWORDS]
    else:
        palabras = [p for p in palabras if p]
    return palabras

class CorpusFactory:
    """
    Iterador re-entrante sobre el corpus raw_es.
    Gensim necesita dos pasadas (build_vocab + train),
    por eso cada llamada a __iter__ genera un iterador nuevo.
    """
    def __init__(self, directorio, archivos, eliminar_stopwords=True):
        self.directorio = directorio
        self.archivos = archivos
        self.eliminar_stopwords = eliminar_stopwords

    def __iter__(self):
        for nombre_archivo in self.archivos:
            ruta = os.path.join(self.directorio, nombre_archivo)
            with open(ruta, "r", encoding="latin-1", errors="ignore") as f:
                for linea in f:
                    palabras = preprocesar_linea(linea, self.eliminar_stopwords)
                    if palabras:
                        yield palabras

In [29]:
corpus_con_sw = CorpusFactory(corpus_dir, corpus_files, eliminar_stopwords=False)
corpus_sin_sw = CorpusFactory(corpus_dir, corpus_files, eliminar_stopwords=True)

N = 20  # Cambia este número para ver más o menos frases

# Verificación visual
print("-- CON stopwords --")
for i, frase in enumerate(corpus_con_sw):
    print(f"  [{i+1}] {frase[:10]}")
    if i == N - 1: break

print("\n-- SIN stopwords --")
for i, frase in enumerate(corpus_sin_sw):
    print(f"  [{i+1}] {frase[:10]}")
    if i == N - 1: break

-- CON stopwords --
  [1] ['acontecimientos']
  [2] ['nacimientos']
  [3] ['fallecimientos']
  [4] ['fulgencio', 'de', 'écija', 'santo', 'español']
  [5] ['erquinoaldo', 'mayordomo', 'franco', 'de', 'palacio', 'de', 'neustria']
  [6] ['acontecimientos']
  [7] ['nacimientos']
  [8] ['egilona', 'última', 'reina', 'visigoda', 'de', 'hispania']
  [9] ['fallecimientos']
  [10] ['acontecimientos']
  [11] ['fin', 'del', 'califato', 'perfecto', 'los', 'omeyas', 'en', 'el', 'poder', 'califato']
  [12] ['nacimientos']
  [13] ['fallecimientos']
  [14] ['acontecimientos']
  [15] ['los', 'omeyas', 'acceden', 'al', 'califato']
  [16] ['nacimientos']
  [17] ['fallecimientos']
  [18] ['mundo', 'islámico', 'alí', 'murió', 'traicionado', 'por', 'los', 'suyos', 'y', 'asesinado']
  [19] ['acontecimientos']
  [20] ['nacimientos']

-- SIN stopwords --
  [1] ['acontecimientos']
  [2] ['nacimientos']
  [3] ['fallecimientos']
  [4] ['fulgencio', 'écija', 'santo', 'español']
  [5] ['erquinoaldo', 'mayordomo', '

## 3. Modelos Word2Vec con raw_es

Word2Vec es un modelo auto-supervisado, por lo que no dispone de un conjunto de validación 
con etiquetas. Esto impide aplicar directamente un GridSearch estándar (que requiere `.fit(X, y)`).

Como solución, realizamos un **grid search manual**: entrenamos distintas combinaciones de 
hiperparámetros sobre una muestra pequeña del corpus, evaluamos cada modelo con la correlación 
de Spearman sobre Multi-SimLex, y finalmente ampliamos el mejor modelo con más datos.

Además, para cada configuración entrenaremos **dos variantes**: con y sin stopwords.

In [30]:
from sklearn.model_selection import KFold
from itertools import product

def evaluar_modelo(model, pares_df, annotator_cols, n_splits=5):
    """
    Calcula la correlación de Spearman sobre los pares de Multi-SimLex:
      - Por cada anotador individual
      - Sobre la media (SPA_AVG)
    Usa K-Fold Cross Validation y retorna media y std de las correlaciones.
    """
    # Filtrar pares donde ambas palabras están en el vocabulario
    mascara = pares_df.apply(
        lambda r: r['SPA_W1'] in model.wv and r['SPA_W2'] in model.wv,
        axis=1
    )
    df_valid = pares_df[mascara].reset_index(drop=True)
    n_oov = len(pares_df) - len(df_valid)
    print(f"  Pares válidos: {len(df_valid)}/{len(pares_df)} — OOV: {n_oov}")

    if len(df_valid) < n_splits:
        return np.nan, np.nan

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

    # ── Correlación sobre SPA_AVG ────────────────────────────────
    corrs_avg = []
    for _, idx_test in kf.split(df_valid):
        fold = df_valid.iloc[idx_test]
        sims_modelo = [
            model.wv.similarity(r['SPA_W1'], r['SPA_W2'])
            for _, r in fold.iterrows()
        ]
        corr, _ = spearmanr(sims_modelo, fold['SPA_AVG'].tolist())
        corrs_avg.append(corr)

    media_avg = np.mean(corrs_avg)
    std_avg   = np.std(corrs_avg)

    # ── Correlación por anotador individual ─────────────────────
    corrs_anotadores = {}
    for col in annotator_cols:
        corrs_col = []
        for _, idx_test in kf.split(df_valid):
            fold = df_valid.iloc[idx_test]
            sims_modelo = [
                model.wv.similarity(r['SPA_W1'], r['SPA_W2'])
                for _, r in fold.iterrows()
            ]
            corr, _ = spearmanr(sims_modelo, fold[col].tolist())
            corrs_col.append(corr)
        corrs_anotadores[col] = np.mean(corrs_col)

    print(f"  Spearman (SPA_AVG): {media_avg:.4f} ± {std_avg:.4f}")
    print(f"  Spearman por anotador: min={min(corrs_anotadores.values()):.4f} "
          f"max={max(corrs_anotadores.values()):.4f} "
          f"media={np.mean(list(corrs_anotadores.values())):.4f}")

    return media_avg, std_avg, corrs_anotadores

In [31]:
import copy

def truncar_vectores(model, nueva_dim):
    """
    Devuelve una copia del modelo con los vectores truncados
    a las primeras nueva_dim dimensiones.
    Permite comparar el efecto de la dimensionalidad sin reentrenar.
    """
    model_copia = copy.deepcopy(model)
    model_copia.wv.vectors = model_copia.wv.vectors[:, :nueva_dim]
    model_copia.vector_size = nueva_dim
    return model_copia

### Word2Vec: CBOW

L'arquitectura *Continuous Bag of Words* de Word2Vec agafa el context i prediu la paraula central. En aquesta arquitectura, l'ordre de les paraules no importa, només quines paraules hi ha. Un exemple clàssic és quan volem completar una frase que li falta una paraula segons el context que ens dóna les altres paraules:
 - "El gat seu sobre la ___"

### Word2Vec: Skip-gram

L'estructura *Skip-gram* de Word2Vec fa el procés invers a CBOW: ara sabem la paraula central, així que volem predir les paraules que estiguin al voltant (el context). Aquest model prediu per a cada posició de la finestra de manera independent, per tant pot generar molts exemples possibles, i això fa que el seu entrenament sigui més lent que CBOW. L'avantatge és que aprèn millor les paraules menys freqüents: com que genera molts exemples per cada paraula, fins i tot les poc freqüents es veuen prou vegades per aprendre una bona representació.

In [32]:
# Comprobamos los nombres reales de las columnas
print(spa.columns.tolist())

['ID', 'Word 1', 'Word 2', 'PoS', 'Annotator 1', 'Annotator 2', 'Annotator 3', 'Annotator 4', 'Annotator 5', 'Annotator 6', 'Annotator 7', 'Annotator 8', 'Annotator 9', 'Annotator 10', 'SPA_AVG']


In [33]:
# ============================================================
# GRID SEARCH MANUAL — Word2Vec (CBOW y Skip-gram)
# Experimentos:
#   1. Con y sin stopwords
#   2. Diferente cantidad de archivos del corpus
#   3. Diferentes tamaños de vector (100, 300) + truncado a 100
# ============================================================

param_grid = {
    'vector_size': [100, 300],
    'window':      [3, 5],
    'min_count':   [5, 10],
    'epochs':      [5, 10],
}

claves        = list(param_grid.keys())
combinaciones = list(product(*param_grid.values()))

# Recuperar columnas de anotadores del dataset original
annotator_cols = [c for c in spa.columns if c.startswith('Annotator')]

N_ARCHIVOS_MUESTRA = 5    # Para el grid search
N_ARCHIVOS_FINAL   = 20   # Para ampliar el mejor modelo

# Diferentes tamaños de corpus para el experimento de cantidad de datos
EXPERIMENTO_N_ARCHIVOS = [5, 20, 57]

resultados_globales = []  # Guardaremos todos los resultados aquí

for sg, nombre_arq in [(0, "CBOW"), (1, "Skip-gram")]:
    for eliminar_sw, sufijo_sw in [(True, "sin_sw"), (False, "con_sw")]:

        nombre_modelo = f"w2v_{nombre_arq.lower().replace('-','_')}_{sufijo_sw}.model"

        print(f"\n{'='*65}")
        print(f"  {nombre_arq} — {sufijo_sw.replace('_', ' ')}")
        print(f"{'='*65}")

        # ── ¿Ya existe el modelo entrenado? ─────────────────────
        if os.path.exists(nombre_modelo):
            print(f"  Modelo ya existente, cargando {nombre_modelo}...")
            mejor_modelo = Word2Vec.load(nombre_modelo)
            print(f"  Cargado ✓ — vocabulario: {len(mejor_modelo.wv):,} palabras")

        else:
            # ── FASE 1: MUESTRA EN RAM ───────────────────────────
            print(f"  Cargando muestra ({N_ARCHIVOS_MUESTRA} archivos)...")
            corpus_muestra = list(
                CorpusFactory(corpus_dir, corpus_files[:N_ARCHIVOS_MUESTRA],
                              eliminar_stopwords=eliminar_sw)
            )
            print(f"  Frases en la muestra: {len(corpus_muestra):,}")

            # ── FASE 2: GRID SEARCH ──────────────────────────────
            print(f"\n  Total combinaciones: {len(combinaciones)}\n")

            resultados   = []
            mejor_modelo = None
            mejor_score  = -np.inf

            for i, combo in enumerate(combinaciones):
                params = dict(zip(claves, combo))
                print(f"  [{i+1}/{len(combinaciones)}] {params}")

                modelo = Word2Vec(
                    sentences=corpus_muestra,
                    vector_size=params['vector_size'],
                    window=params['window'],
                    min_count=params['min_count'],
                    sg=sg,
                    workers=4,
                    epochs=params['epochs'],
                    seed=42
                )

                media, std, corrs_anot = evaluar_modelo(
                    modelo, multi_simlex, annotator_cols, n_splits=5
                )
                resultados.append({
                    'arquitectura':  nombre_arq,
                    'stopwords':     sufijo_sw,
                    **params,
                    'spearman_mean': media,
                    'spearman_std':  std
                })

                if media > mejor_score:
                    mejor_score  = media
                    mejor_modelo = modelo

            # ── RESULTADOS GRID SEARCH ───────────────────────────
            resultados_df = pd.DataFrame(resultados).sort_values(
                'spearman_mean', ascending=False
            )
            print(f"\n  Ranking ({nombre_arq} — {sufijo_sw}):")
            print(resultados_df.to_string(index=False))

            mejores = resultados_df.iloc[0]
            print(f"\n  Mejores hiperparámetros:")
            print(f"    vector_size : {int(mejores['vector_size'])}")
            print(f"    window      : {int(mejores['window'])}")
            print(f"    min_count   : {int(mejores['min_count'])}")
            print(f"    epochs      : {int(mejores['epochs'])}")
            print(f"    Spearman    : {mejores['spearman_mean']:.4f} ± {mejores['spearman_std']:.4f}")

            # ── FASE 3: AMPLIAR CON MÁS DATOS ───────────────────
            print(f"\n  Ampliando con {N_ARCHIVOS_FINAL} archivos...")
            corpus_final = list(
                CorpusFactory(corpus_dir, corpus_files[:N_ARCHIVOS_FINAL],
                              eliminar_stopwords=eliminar_sw)
            )
            mejor_modelo.build_vocab(corpus_final, update=True)
            mejor_modelo.train(
                corpus_final,
                total_examples=len(corpus_final),
                epochs=mejor_modelo.epochs
            )
            mejor_modelo.save(nombre_modelo)
            print(f"  Modelo guardado como '{nombre_modelo}' ✓")

        # ── EXPERIMENTO: EFECTO CANTIDAD DE CORPUS ───────────────
        print(f"\n  Experimento: efecto de la cantidad de corpus")
        for n_arch in EXPERIMENTO_N_ARCHIVOS:
            n_arch = min(n_arch, len(corpus_files))
            corpus_exp = list(
                CorpusFactory(corpus_dir, corpus_files[:n_arch],
                              eliminar_stopwords=eliminar_sw)
            )
            modelo_exp = Word2Vec(
                sentences=corpus_exp,
                vector_size=mejor_modelo.vector_size,
                window=mejor_modelo.window,
                min_count=mejor_modelo.min_count,
                sg=sg,
                workers=4,
                epochs=mejor_modelo.epochs,
                seed=42
            )
            media, std, _ = evaluar_modelo(
                modelo_exp, multi_simlex, annotator_cols, n_splits=5
            )
            print(f"    {n_arch:2d} archivos → Spearman: {media:.4f} ± {std:.4f}")
            resultados_globales.append({
                'arquitectura': nombre_arq,
                'stopwords':    sufijo_sw,
                'n_archivos':   n_arch,
                'vector_size':  mejor_modelo.vector_size,
                'spearman':     media
            })

        # ── EXPERIMENTO: TRUNCADO DE VECTORES A 100d ─────────────
        if mejor_modelo.vector_size > 100:
            print(f"\n  Experimento: truncado de vectores a 100d")
            modelo_trunc = truncar_vectores(mejor_modelo, 100)
            media_trunc, std_trunc, _ = evaluar_modelo(
                modelo_trunc, multi_simlex, annotator_cols, n_splits=5
            )
            print(f"    {mejor_modelo.vector_size}d original → Spearman: "
                  f"{media_trunc:.4f} ± {std_trunc:.4f} (truncado a 100d)")

        # ── GUARDAR REFERENCIA AL MODELO ─────────────────────────
        if nombre_arq == "CBOW" and sufijo_sw == "sin_sw":
            w2v_cbow_sin_sw = mejor_modelo
        elif nombre_arq == "CBOW" and sufijo_sw == "con_sw":
            w2v_cbow_con_sw = mejor_modelo
        elif nombre_arq == "Skip-gram" and sufijo_sw == "sin_sw":
            w2v_sg_sin_sw = mejor_modelo
        else:
            w2v_sg_con_sw = mejor_modelo


  CBOW — sin sw
  Cargando muestra (5 archivos)...
  Frases en la muestra: 531,872

  Total combinaciones: 16

  [1/16] {'vector_size': 100, 'window': 3, 'min_count': 5, 'epochs': 5}
  Pares válidos: 1556/1888 — OOV: 332
  Spearman (SPA_AVG): 0.2598 ± 0.0283
  Spearman por anotador: min=0.1475 max=0.2508 media=0.1941
  [2/16] {'vector_size': 100, 'window': 3, 'min_count': 5, 'epochs': 10}
  Pares válidos: 1556/1888 — OOV: 332
  Spearman (SPA_AVG): 0.3125 ± 0.0393
  Spearman por anotador: min=0.1839 max=0.3005 media=0.2319
  [3/16] {'vector_size': 100, 'window': 3, 'min_count': 10, 'epochs': 5}
  Pares válidos: 1474/1888 — OOV: 414
  Spearman (SPA_AVG): 0.2689 ± 0.0315
  Spearman por anotador: min=0.1562 max=0.2644 media=0.2017
  [4/16] {'vector_size': 100, 'window': 3, 'min_count': 10, 'epochs': 10}
  Pares válidos: 1474/1888 — OOV: 414
  Spearman (SPA_AVG): 0.3158 ± 0.0291
  Spearman por anotador: min=0.1812 max=0.3114 media=0.2356
  [5/16] {'vector_size': 100, 'window': 5, 'min_coun

KeyboardInterrupt: 

In [ ]:
# ============================================================
# RESUMEN VISUAL — Efecto cantidad de corpus
# ============================================================
import matplotlib.pyplot as plt

df_exp = pd.DataFrame(resultados_globales)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Efecto de la cantidad de corpus — Word2Vec", fontsize=13)

for ax, sw in zip(axes, ["sin_sw", "con_sw"]):
    subset = df_exp[df_exp['stopwords'] == sw]
    for arq, color in [("CBOW", "#2196F3"), ("Skip-gram", "#FF9800")]:
        datos = subset[subset['arquitectura'] == arq].sort_values('n_archivos')
        ax.plot(datos['n_archivos'], datos['spearman'],
                'o-', label=arq, color=color, linewidth=2, markersize=7)
    ax.set_title(sw.replace('_', ' '))
    ax.set_xlabel("Número de archivos del corpus")
    ax.set_ylabel("Spearman r (Multi-SimLex)")
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("efecto_corpus.png", dpi=150, bbox_inches='tight')
plt.show()
print("Gráfico guardado como 'efecto_corpus.png'")

## 4. Models FastText